# 03 · Degraded NE-US (PLAN_UPDATE_v2 §2.5)
Separates the effect of a sparse network, station outages and a coarse grid from the effect of the tropical climate,
using the paper's NE-US data. Results go to `docs/results_degraded_neus.md`.

Needs on `$DATA_ROOT`:
- `neus/`: the Zenodo tree from notebook 01 (filtered 0.9 station files, `ERA5/Processed/era5_{year}_-83_-65_37_49.nc`).
- `brazil/`: the notebook 02 outputs (region shapefile, all-station shapefile and processed files), only to measure
  the Brazil outage statistics that the gap runs copy.

Runtime: GPU. Each run is a 20-epoch Transformer + ERA5 Nearest, lead 1 h (about 1 h on an A100 for 358 stations,
less for 26). Runs whose test outputs already exist are skipped, so after a Colab disconnect just run all cells again.

In [ ]:
# Setup. Colab: clones/updates the repo, installs requirements, mounts Drive.
# PC: start Jupyter anywhere inside the repo and set the DATA_ROOT environment variable first.
import os, subprocess, sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    repo = Path('/content/LocalizedWeatherBR')
    if repo.exists():
        subprocess.run(['git', '-C', str(repo), 'pull', '-q'], check=True)
    else:
        subprocess.run(['git', 'clone', '-q', 'https://github.com/joancsz/LocalizedWeatherBR.git', str(repo)], check=True)
    os.chdir(repo)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_ROOT = '/content/drive/MyDrive/LocalizedWeather'
else:
    os.chdir(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'PLAN_BRAZIL.md').exists()))
    DATA_ROOT = os.environ.get('DATA_ROOT') or sys.exit('set DATA_ROOT (folder holding neus/ and brazil/) before starting Jupyter')

os.environ['DATA_ROOT'] = DATA_ROOT  # configs read ${DATA_ROOT}
PY = sys.executable                   # !{PY} runs the scripts with this kernel's Python on Colab and Windows alike
print('repo:', os.getcwd(), '| DATA_ROOT:', DATA_ROOT)

## Unit tests

In [ ]:
!{PY} -m pytest -q tests/

## Brazil outage statistics (gap targets)
Over the 26 stations kept at control ratio 0.6, 2019–2023: the fraction of hours with real `u`, and the mean length
of an outage (a run of hours without `u`). The gap runs inject outages into NE-US until it matches both.

In [ ]:
import numpy as np, geopandas as gpd, xarray as xr, pandas as pd
sys.path.insert(0, 'Source')
import Utils.Paths as Paths

BR = Path(DATA_ROOT) / 'brazil'
START, END, BR_RATIO = 2019, 2023, 0.6
shp = BR / 'Shapefiles/Regions/ne_brazil_buffered.shp'
lon_lo, lat_lo, lon_up, lat_up = gpd.read_file(shp).bounds.iloc[0]
br_name = Paths.region_file_name(lon_lo, lon_up, lat_lo, lat_up, shp)
stations = gpd.read_file(Paths.station_file(BR, START, END, br_name))
n_hours = len(pd.date_range(f'{START}-01-01', f'{END}-12-31 23:00', freq='h'))
kept = (stations.num >= BR_RATIO * n_hours).to_numpy()
assert kept.sum() == 26, f'{kept.sum()} Brazil stations at {BR_RATIO}, expected 26 (docs/repro_log.md P3)'

real, gaps = [], []
for y in range(START, END + 1):
    with xr.open_dataset(Paths.madis_processed_file(BR, START, END, y, br_name)) as ds:
        is_real = ds.u_is_real.values[kept]
    real.append(is_real.mean())
    for row in is_real:
        d = np.diff(np.concatenate([[1], row.astype(int), [1]]))
        gaps.extend(np.flatnonzero(d == 1) - np.flatnonzero(d == -1))
BR_REAL = round(float(np.mean(real)), 3)
BR_GAP_H = int(round(np.mean(gaps)))
print('Brazil u real fraction per year:', np.round(real, 3))
print(f'target_real_fraction = {BR_REAL}, mean_gap_hours = {BR_GAP_H} (from {len(gaps)} outages, '
      f'median {int(np.median(gaps))} h)')

## Runs
Each run is `configs/neus_deg_base.yaml` plus its degradation keys; the generated config is saved next to its outputs
(`$DATA_ROOT/neus/ModelOutputs/<run>/config.yaml`). In priority order: the first five are the plan's minimal design,
`sparse_gaps` is the Brazil-like case (26 stations and Brazil outages on the native 0.25° grid, as Brazil's ERA5 is
also 0.25°), and `coarse_050` is the intermediate grid.

In [ ]:
import yaml
GAPS = dict(target_real_fraction=BR_REAL, mean_gap_hours=BR_GAP_H, seed=0)
RUNS = {
    'base':        {},
    'sparse':      dict(station_keep_n=26),
    'gaps':        dict(gap_injection=GAPS),
    'coarse_100':  dict(era5_coarsen_deg=1.0),
    'all':         dict(station_keep_n=26, gap_injection=GAPS, era5_coarsen_deg=1.0),
    'sparse_gaps': dict(station_keep_n=26, gap_injection=GAPS),
    'coarse_050':  dict(era5_coarsen_deg=0.5),
}
# lead sweep: the factors that matter at 6 h, and base out to the paper's 48 h
for l, names in ((6, ['base', 'coarse_100', 'sparse_gaps', 'all']), (24, ['base']), (48, ['base'])):
    RUNS.update({f'{n}_l{l}': {**RUNS[n], 'lead_hrs': l} for n in names})
base = yaml.safe_load(open('configs/neus_deg_base.yaml'))
OUT = Path(DATA_ROOT) / 'neus' / 'ModelOutputs'
for name, extra in RUNS.items():
    cfg = {**base, **extra, 'output_saving_path': f'ModelOutputs/neus_deg_{name}'}
    (OUT / f'neus_deg_{name}').mkdir(parents=True, exist_ok=True)
    yaml.safe_dump(cfg, open(OUT / f'neus_deg_{name}' / 'config.yaml', 'w'), sort_keys=False)
print('\n'.join(f'{k}: {v}' for k, v in RUNS.items()))

In [ ]:
import torch
assert torch.cuda.is_available(), 'switch to a GPU runtime'
print(torch.cuda.get_device_name(0))

In [ ]:
# a run is done when its test predictions exist; delete that file to rerun it
for name in RUNS:
    run_dir = OUT / f'neus_deg_{name}'
    if (run_dir / 'Preds_CUSTOM_min.pkl').exists():
        print('skip', name)
        continue
    print('=== run', name, flush=True)
    cfg_path = run_dir / 'config.yaml'
    !{PY} Source/run.py --config "{cfg_path}"
    if _exit_code:
        raise RuntimeError(f'run {name} failed (exit {_exit_code})')

## Test metrics, copy into `docs/results_degraded_neus.md`
Per lead, every run is scored on the stations of the sparsest run (26) and on the entries that are real in every run
(`scripts/eval/common_set.py`), so rows are directly comparable. Then ERA5 and persistence for each `base*` run.

In [ ]:
import json
sys.path.insert(0, 'scripts/eval')
import common_set
from IPython.display import Markdown, display
by_lead = {}
for name in RUNS:
    run_dir = OUT / f'neus_deg_{name}'
    if (run_dir / 'Preds_CUSTOM_min.pkl').exists():
        by_lead.setdefault(json.load(open(run_dir / 'params.json'))['lead_hrs'], []).append(run_dir)
for lead, dirs in sorted(by_lead.items()):
    ref = min(dirs, key=lambda d: len(common_set.station_xy(d)))
    rows = ['| run | wind-vector error (m/s) | T RMSE (K) | Td RMSE (K) |', '|---|---|---|---|']
    for n, r in common_set.common_scores(dirs, ref).items():
        f = lambda k: '{:.3f} [{:.3f}, {:.3f}]'.format(r[k], *r[k + '_ci'])
        rows.append(f"| {n.removeprefix('neus_deg_')} | {f('wind')} | {f('t_rmse')} | {f('td_rmse')} |")
    display(Markdown(f'**Lead {lead} h** ({len(common_set.station_xy(ref))} stations)\n\n' + '\n'.join(rows)))

In [ ]:
for name in RUNS:
    run_dir = OUT / f'neus_deg_{name}'
    if name.split('_l')[0] == 'base' and (run_dir / 'Preds_CUSTOM_min.pkl').exists():
        print('===', name)
        !{PY} scripts/eval/baselines.py --run-dir "{run_dir}"